In [36]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations

# Load and prepare data
path = "/Users/geoffrey/Desktop/Uni/2025/S2/DATA2902/Group_V1_Python/NY_House_Prices.csv"
house_prices = pd.read_csv(path)
house_prices.columns = house_prices.columns.str.replace('.', "")

# Separate training data (Test=0) and test data (Test=1)
train_data = house_prices[house_prices['Test'] == 0].copy()
test_data = house_prices[house_prices['Test'] == 1].copy()

# Define response and predictors
response = 'Price'
predictors = ['LotSize', 'Waterfront', 'Age', 'LandValue', 'NewConstruct',
              'CentralAir', 'FuelType', 'HeatType', 'SewerType', 'LivingArea',
              'PctCollege', 'Bedrooms', 'Fireplaces', 'Bathrooms', 'Rooms']

# Convert categorical variables to dummy variables
train_encoded = pd.get_dummies(train_data[predictors + [response]], 
                                columns=['FuelType', 'HeatType', 'SewerType'],
                                drop_first=True)
predictor_cols = [col for col in train_encoded.columns if col != response]

print(f"Total predictors after encoding: {len(predictor_cols)}\n")

Total predictors after encoding: 20



## selection

In [37]:
import time

# training X and y
X_train = train_encoded[predictor_cols]
y_train = train_encoded[response]

# Handle NaNs
train_encoded = train_encoded.dropna()
X_train = train_encoded[predictor_cols]
y_train = train_encoded[response]

# force all bool columns to int (0/1)
X_train = X_train.astype(float)
y_train = y_train.astype(float)

# print(f"Starting exhaustive search on {len(predictor_cols)} predictors...")

# # --- Exhaustive Search ---
# # store our results in a list of dictionaries
# results = []
# start_time = time.time()

# # Iterate through every possible number of predictors (k)
# for k in range(1, len(predictor_cols) + 1):
    
#     # Get all combinations of predictors for the current k
#     for combo in combinations(predictor_cols, k):
        
#         # 1. Get the subset of predictors
#         X_subset = X_train[list(combo)]
        
#         # 2. Add a constant (intercept) to the model
#         X_subset_const = sm.add_constant(X_subset)
        
#         # 3. Fit the OLS model
#         model = sm.OLS(y_train, X_subset_const).fit()
        
#         # 4. Store the results
#         results.append({
#             'k': k,
#             'features': combo,
#             'aic': model.aic,
#             'adj_r2': model.rsquared_adj
#         })

# end_time = time.time()
# print(f"--- Search Complete ---")
# print(f"Total models fitted: {len(results)}")
# print(f"Total time: {end_time - start_time:.2f} seconds\n")

# # --- Process Results ---

# # Convert results to a DataFrame for easy sorting
# results_df = pd.DataFrame(results)

# # Find the best model based on lowest AIC
# best_model_aic = results_df.loc[results_df['aic'].idxmin()]

# # Find the best model based on highest Adjusted R-squared
# best_model_adj_r2 = results_df.loc[results_df['adj_r2'].idxmax()]

# # --- Print the Best Models ---

# print("="*50)
# print("Best Model (Lowest AIC)")
# print("="*50)
# print(f"AIC: {best_model_aic['aic']:.2f}")
# print(f"Adj. R-squared: {best_model_aic['adj_r2']:.4f}")
# print(f"Number of Features: {best_model_aic['k']}")
# print("Features:")
# for f in best_model_aic['features']:
#     print(f"  - {f}")
# print("\n")


# print("="*50)
# print("Best Model (Highest Adjusted R-squared)")
# print("="*50)
# print(f"AIC: {best_model_adj_r2['aic']:.2f}")
# print(f"Adj. R-squared: {best_model_adj_r2['adj_r2']:.4f}")
# print(f"Number of Features: {best_model_adj_r2['k']}")
# print("Features:")
# for f in best_model_adj_r2['features']:
#     print(f"  - {f}")
# print("\n")

# #  can now fit these "best" models again to view their full summary
# print("--- Building full summary for the Best AIC Model ---")
# best_features_aic = list(best_model_aic['features'])
# X_best_aic = sm.add_constant(X_train[best_features_aic])
# final_model_aic = sm.OLS(y_train, X_best_aic).fit()

# print(final_model_aic.summary())

--- Search Complete ---
Total models fitted: 1048575
Total time: 1311.54 seconds

==================================================
Best Model (Lowest AIC)
==================================================
AIC: 32187.61
Adj. R-squared: 0.6378
Number of Features: 11
Features:
  - LotSize
  - Waterfront
  - Age
  - LandValue
  - NewConstruct
  - CentralAir
  - LivingArea
  - Bedrooms
  - Bathrooms
  - Rooms
  - HeatType_Hot Air


==================================================
Best Model (Highest Adjusted R-squared)
==================================================
AIC: 32187.61
Adj. R-squared: 0.6378
Number of Features: 11
Features:
  - LotSize
  - Waterfront
  - Age
  - LandValue
  - NewConstruct
  - CentralAir
  - LivingArea
  - Bedrooms
  - Bathrooms
  - Rooms
  - HeatType_Hot Air

## test data + summary

In [38]:
print("="*50)
print("Building Model with Best 11 Selected Features")
print("="*50)

# Define selected list of features
selected_features = [
    'LotSize', 
    'Waterfront', 
    'Age', 
    'LandValue', 
    'NewConstruct',
    'CentralAir', 
    'LivingArea', 
    'Bedrooms', 
    'Bathrooms', 
    'Rooms',
    'HeatType_Hot Air' 
]

# Create the X dataframe
X_model = X_train[selected_features]

# Add a constant (intercept)
X_model_const = sm.add_constant(X_model)

# Fit the OLS model
model_selected = sm.OLS(y_train, X_model_const).fit()

# Print the summary
print(model_selected.summary())

Building Model with Best 11 Selected Features
                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.641
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                     209.0
Date:                Sat, 18 Oct 2025   Prob (F-statistic):          4.23e-277
Time:                        14:34:29   Log-Likelihood:                -16082.
No. Observations:                1300   AIC:                         3.219e+04
Df Residuals:                    1288   BIC:                         3.225e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------

In [39]:
## MSE
import numpy as np
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

test_encoded = pd.get_dummies(test_data[predictors + [response]], 
                              columns=['FuelType', 'HeatType', 'SewerType'],
                              drop_first=True)

# Drop NaNs
test_encoded = test_encoded.dropna() 

# Define X_test and y_test
X_test = test_encoded.reindex(columns=X_train.columns, fill_value=0).astype(float)
y_test = test_encoded[response].astype(float)

#  Create the X_test 
X_test_subset = X_test[selected_features]
X_test_subset_const = sm.add_constant(X_test_subset)

# make predictions
y_pred = model_selected.predict(X_test_subset_const)

#Calculate and print RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test Data RMSE: {rmse:.2f}")

Test Data RMSE: 60970.89
